# Module 11: Animation

Interactive plots hand control to the viewer: they can explore whatever region or point interests them. Animation is a different tool with a different purpose: it controls the time dimension explicitly, advancing the figure frame by frame at a pace you define. This is the right approach when you want to show how a system evolves over time, how a signal builds up, or how a model improves across training iterations, and when the sequence itself carries meaning that a static or interactive figure would lose.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

## 1. The structure of `FuncAnimation`

`matplotlib.animation.FuncAnimation` is the standard Matplotlib tool for frame-by-frame animation. It takes three main ingredients.

The **`init` function** is called once at the start. It sets up the blank canvas: empty plot objects that will be updated later. Think of it as placing a fresh sheet of paper on the drawing board.

The **`update` function** is called once per frame. It receives a frame index (an integer counting up from zero) and modifies the plot objects in place. Think of it as the hand that draws the next line on the paper before the shutter clicks.

The **`frames` argument** controls how many times `update` is called. If you pass an integer `N`, `update` is called with `0, 1, 2, ..., N-1`. You can also pass an iterable (a list, a `range`, or a generator) if you want to supply the frame values directly rather than just a count.

The full call looks like this:

```python
anim = animation.FuncAnimation(
    fig,          # the Figure object to animate
    update,       # function called on every frame
    frames=N,     # how many frames to produce
    init_func=init,  # optional: called before the first frame
    interval=50,  # milliseconds between frames (controls playback speed)
    repeat=True   # whether to loop after the last frame
)
```

`update` must return an iterable of the artists it modified. This is required for Matplotlib to know which parts of the figure need to be redrawn efficiently.

## 2. A minimal first animation: DSC heating curve

A DSC (differential scanning calorimetry) curve records the heat flow into a polymer sample as it is heated at a constant rate. Two features are immediately recognizable to anyone in polymer science: a step change in heat flow at the glass transition temperature $T_g$, where the material transitions from a glassy to a rubbery state, and an exothermic or endothermic peak at the melting temperature $T_m$ for semi-crystalline polymers. Animating this curve as if the instrument were recording in real time gives a direct sense of how the measurement unfolds with temperature.

In [ ]:
# Synthetic DSC curve for a semi-crystalline polymer (e.g. PET-like)
T = np.linspace(30, 280, 500)   # temperature axis in deg C

# Baseline: slight upward slope representing the sensible heat capacity
dsc_baseline = 0.002 * (T - 30)

# Glass transition step: sigmoidal change in heat flow at Tg = 80 C
Tg = 80.0
dsc_tg = 0.18 / (1 + np.exp(-(T - Tg) / 3.0))

# Cold crystallization exotherm: narrow negative peak near 130 C
T_cc = 130.0
dsc_cc = -0.45 * np.exp(-((T - T_cc) ** 2) / (2 * 7.0 ** 2))

# Melting endotherm: broader positive peak near 240 C
T_melt = 240.0
dsc_melt = 0.90 * np.exp(-((T - T_melt) ** 2) / (2 * 10.0 ** 2))

# Full DSC signal
dsc_signal = dsc_baseline + dsc_tg + dsc_cc + dsc_melt

# Add a small amount of noise to simulate real instrument noise
rng = np.random.default_rng(seed=5)
dsc_signal += rng.normal(0, 0.004, len(T))

In [ ]:
# Number of frames: one frame reveals every 5 data points
step = 5
n_frames = len(T) // step

fig_dsc, ax_dsc = plt.subplots(figsize=(8, 4))
ax_dsc.set_xlim(T[0], T[-1])
ax_dsc.set_ylim(dsc_signal.min() - 0.05, dsc_signal.max() + 0.1)
ax_dsc.set_xlabel('Temperature (°C)')
ax_dsc.set_ylabel('Heat flow (W g⁻¹, endo up)')
ax_dsc.set_title('DSC heating curve (10 °C min⁻¹)')

# Annotate the known thermal events at fixed positions
ax_dsc.axvline(Tg,    color='steelblue',  linewidth=0.8, linestyle=':', alpha=0.7)
ax_dsc.axvline(T_cc,  color='seagreen',   linewidth=0.8, linestyle=':', alpha=0.7)
ax_dsc.axvline(T_melt,color='firebrick',  linewidth=0.8, linestyle=':', alpha=0.7)
ax_dsc.text(Tg + 2,    dsc_signal.max() + 0.05, f'$T_g$ = {Tg:.0f} °C', fontsize=8, color='steelblue')
ax_dsc.text(T_cc + 2,  dsc_signal.max() + 0.05, f'$T_{{cc}}$ = {T_cc:.0f} °C', fontsize=8, color='seagreen')
ax_dsc.text(T_melt + 2,dsc_signal.max() + 0.05, f'$T_m$ = {T_melt:.0f} °C', fontsize=8, color='firebrick')

# The line artist that will be updated each frame
(dsc_line,) = ax_dsc.plot([], [], color='black', linewidth=1.5)

# Marker showing the current recording position
(dsc_dot,) = ax_dsc.plot([], [], 'o', color='firebrick', markersize=5, zorder=5)

plt.tight_layout()

def dsc_init():
    dsc_line.set_data([], [])
    dsc_dot.set_data([], [])
    return dsc_line, dsc_dot

def dsc_update(frame):
    # Reveal data up to the current frame index
    end_idx = (frame + 1) * step
    dsc_line.set_data(T[:end_idx], dsc_signal[:end_idx])
    # Move the dot to the leading edge of the curve
    dsc_dot.set_data([T[end_idx - 1]], [dsc_signal[end_idx - 1]])
    return dsc_line, dsc_dot

anim_dsc = animation.FuncAnimation(
    fig_dsc,
    dsc_update,
    frames=n_frames,
    init_func=dsc_init,
    interval=40,    # 40 ms per frame = 25 fps
    repeat=False
)

plt.close()   # prevent a static figure from appearing alongside the animation
print(f'Animation ready: {n_frames} frames at 25 fps = {n_frames * 40 / 1000:.1f} s')

Calling `plt.close()` after creating the animation suppresses the static figure that Matplotlib would otherwise display when the cell runs. The animation object `anim_dsc` is separate from the figure and is displayed in its own output cell further below. The `init` function clears both artists to empty data; this is important when `repeat=True` is set, because it ensures the figure is wiped clean before the loop restarts.

## 3. Controlling playback

Three arguments to `FuncAnimation` control how the animation plays:

`interval` is the delay between frames in milliseconds. At `interval=40` the animation runs at 25 frames per second; at `interval=100` it runs at 10 fps. Lower values feel faster; the actual smoothness depends on how quickly `update` can execute.

`frames` sets the total frame count. Passing an integer `N` calls `update` with `0` through `N-1`. Passing a list or range gives you explicit control over which values are passed to `update`, which is useful when frames do not map cleanly onto a simple count.

`repeat` controls looping. `repeat=True` (the default) loops the animation indefinitely; `repeat=False` stops after the last frame. A related argument, `repeat_delay`, adds a pause in milliseconds between repetitions when looping is on.

In [ ]:
# Demonstrating interval and repeat: a bouncing sine wave
fig_pb, ax_pb = plt.subplots(figsize=(6, 3))
x_pb = np.linspace(0, 2 * np.pi, 300)
ax_pb.set_xlim(0, 2 * np.pi)
ax_pb.set_ylim(-1.2, 1.2)
ax_pb.set_xlabel('x')
ax_pb.set_ylabel('sin(x + phase)')
ax_pb.set_title('Playback control: interval=30 ms, repeat=True')
(pb_line,) = ax_pb.plot([], [], color='steelblue', linewidth=2)
plt.tight_layout()

def pb_update(frame):
    phase = frame * (2 * np.pi / 60)   # one full cycle over 60 frames
    pb_line.set_data(x_pb, np.sin(x_pb + phase))
    return (pb_line,)

anim_pb = animation.FuncAnimation(
    fig_pb, pb_update,
    frames=60,
    interval=30,        # fast: 30 ms per frame
    repeat=True,        # loop continuously
    repeat_delay=500    # 0.5 s pause before repeating
)

plt.close()
print('Playback demo ready')

## 4. Saving animations: GIF and MP4

Matplotlib can write animations to file through writer backends. `PillowWriter` produces animated GIFs and requires only the `Pillow` library, which is available in Colab by default. `FFMpegWriter` produces MP4 files and requires FFmpeg to be installed on the system. Install it in Colab with the cell below.

In [ ]:
#!apt-get install ffmpeg -qq

In [ ]:
# Save the DSC animation as a GIF using PillowWriter
gif_writer = animation.PillowWriter(fps=25)
anim_dsc.save('./result/dsc_curve.gif', writer=gif_writer, dpi=100)
print('Saved: dsc_curve.gif')

# Save as MP4 using FFMpegWriter
#mp4_writer = animation.FFMpegWriter(fps=25, bitrate=1200)
#anim_dsc.save('./result/dsc_curve.mp4', writer=mp4_writer, dpi=100)
#print('Saved: dsc_curve.mp4')

Both files will appear in the Colab file panel and can be downloaded from there. GIF is broadly compatible and embeds directly in HTML documents and slide decks, but has a 256-color palette limit and can produce large file sizes for long animations. MP4 supports full color and compresses far more efficiently for longer content; it is the better choice when file size matters or when the animation will be included in a video presentation.

## 5. Displaying the animation inline in Colab

`anim.to_jshtml()` converts the animation to a self-contained HTML string with embedded JavaScript playback controls. Wrapping it in `IPython.display.HTML` and returning it from a cell renders the interactive player directly in the notebook output: you get play, pause, step, and speed controls without saving any file.

In [ ]:
# Display the DSC animation inline with interactive controls
HTML(anim_dsc.to_jshtml())

In [ ]:
# Display the playback control demo inline
HTML(anim_pb.to_jshtml())

The `to_jshtml()` player embeds all frames as images, so it works in any environment that renders HTML: Jupyter, Colab, and exported notebooks opened in a browser. The tradeoff is file size: for long animations with many frames, the embedded image data can make the notebook large. For those cases, saving to MP4 and displaying with a video element is more practical.

## 6. Dual-panel animation: ANN training progression

A common use of animation in machine learning is showing how a model's predictions improve as training proceeds. The setup below trains a small neural network on noisy data sampled from a known quartic function. The network has three fully-connected layers with Sigmoid activations and is trained with the Adam optimizer for 1 000 epochs. Every 10 epochs we save a snapshot of the network's predictions so the animation can replay the full training trajectory.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

def true_fn(x):
    return -18*x**4 + 35.2*x**3 - 21*x**2 + 4.4*x

n_points = 120
x_data = np.random.beta(0.8, 0.8, n_points)
y_data = true_fn(x_data) + np.random.normal(0, 0.05, n_points)

x_plot = np.linspace(0, 1, 200)
y_true_plot = true_fn(x_plot)

x_tensor = torch.FloatTensor(x_data.reshape(-1, 1))
y_tensor = torch.FloatTensor(y_data.reshape(-1, 1))
x_plot_tensor = torch.FloatTensor(x_plot.reshape(-1, 1))

dataset = TensorDataset(x_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.Sigmoid(),
            nn.Linear(64, 32),
            nn.Sigmoid(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

total_epochs = 1000
save_interval = 10
snapshots = []
loss_history = []

for epoch in range(total_epochs):
    epoch_loss = 0.0
    batch_count = 0
    for inputs, targets in dataloader:
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count

    if epoch % save_interval == 0 or epoch == total_epochs - 1:
        model.eval()
        with torch.no_grad():
            y_plot_pred = model(x_plot_tensor).numpy().flatten()
        model.train()
        snapshots.append((epoch, y_plot_pred))
        loss_history.append(avg_loss)
        if epoch % 200 == 0:
            print(f'Epoch {epoch:4d}  |  MSE = {avg_loss:.5f}')

loss_history = np.array(loss_history)
save_epochs = [s[0] for s in snapshots]
n_saved = len(snapshots)
print(f'\nTraining complete: {n_saved} snapshots saved')


# ── CODE CELL ──────────────────────────────────────────────────────────────

fig_ann, (ax_fit, ax_loss) = plt.subplots(
    2, 1, figsize=(8, 7),
    gridspec_kw={'height_ratios': [5, 2]},
    sharex=False
)
fig_ann.subplots_adjust(hspace=0.35)

ax_fit.scatter(x_data, y_data, s=14, color='steelblue',
               alpha=0.5, zorder=2, label='Training data')
ax_fit.plot(x_plot, y_true_plot, 'k--', linewidth=1.5, label='True function')
(fit_line,) = ax_fit.plot([], [], color='firebrick', linewidth=2.0,
                          zorder=3, label='NN fit')

ax_fit.set_xlim(-0.05, 1.05)
ax_fit.set_ylim(-0.2, 1.1)
ax_fit.set_xlabel('x')
ax_fit.set_ylabel('y')
ax_fit.legend(fontsize=8, loc='upper left')

epoch_text = ax_fit.text(
    0.97, 0.05, '', transform=ax_fit.transAxes,
    fontsize=9, ha='right', va='bottom',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7)
)

ax_loss.plot(save_epochs, loss_history, color='gray', linewidth=1.0, alpha=0.5)
(loss_dot,) = ax_loss.plot([], [], 'o', color='firebrick',
                           markersize=6, zorder=5)
ax_loss.set_yscale('log')
ax_loss.set_xlim(0, total_epochs)
ax_loss.set_ylim(loss_history.min() * 0.5, loss_history.max() * 2)
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('MSE Loss')

plt.tight_layout()

def ann_init():
    fit_line.set_data([], [])
    loss_dot.set_data([], [])
    epoch_text.set_text('')
    return fit_line, loss_dot, epoch_text

def ann_update(frame):
    ep, y_pred = snapshots[frame]
    fit_line.set_data(x_plot, y_pred)
    loss_dot.set_data([save_epochs[frame]], [loss_history[frame]])
    epoch_text.set_text(f'Epoch {ep + 1}')
    return fit_line, loss_dot, epoch_text

anim_ann = animation.FuncAnimation(
    fig_ann,
    ann_update,
    frames=n_saved,
    init_func=ann_init,
    interval=150,
    repeat=True,
    repeat_delay=1500
)

plt.close()
print(f'Animation ready: {n_saved} frames')

In [ ]:
HTML(anim_ann.to_jshtml())

Watch the top panel: the red curve starts nearly flat — the network hasn ot yet learned any structure — and gradually acquires the shape of the true function as training proceeds. The bottom panel tracks the corresponding MSE on a log scale; the red dot shows exactly where in training the current frame sits. The loss drops steeply during the first few hundred epochs as the network captures the broad curvature, then declines more slowly as it refines the fit in the data-sparse interior.

## Training, energy, and conformational space

There is a physical analogy worth holding onto here. During thermal annealing, a polymer chain explores its conformational space through thermally activated motion, gradually settling into lower-energy configurations as temperature decreases and thermal fluctuations are reduced. A neural network during gradient descent does something structurally similar: it explores parameter space step by step, following the gradient of the loss function downhill, and converges toward a configuration (a set of weights) that represents a lower-energy state in the optimization landscape. In both cases, the system starts disordered, the driving force is a scalar quantity that must decrease (free energy or loss), and the process is path-dependent: the final configuration depends on the trajectory, not just the destination.